# ExDark: CLAHE + Bilateral Denoise + YOLOv8 (Kaggle)

Notebook cho một kịch bản thực nghiệm thống nhất:

`ExDark tối -> CLAHE trên kênh L của LAB -> bilateral denoise -> YOLOv8n`

- CLAHE theo cấu hình baseline của repo NMH: `clipLimit=2.0`, `tileGridSize=(8, 8)`.
- Khử nhiễu bằng chính `src/create_denoised_dataset.py` trên GitHub: `d=5`, `sigmaColor=25`, `sigmaSpace=25`.
- YOLOv8: `yolov8n.pt`, `epochs=40`, `batch=16`, `imgsz=640`, `patience=15`, `seed=42`.

> Kaggle: bật **GPU** và **Internet** trước khi Run All. Mọi đầu ra nằm trong `/kaggle/working`.

In [ ]:
!pip install -q -U ultralytics opencv-python-headless pyyaml tqdm matplotlib

from pathlib import Path
import json, os, shutil, subprocess, sys
import cv2
import matplotlib.pyplot as plt
import torch
import yaml
from tqdm.auto import tqdm

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Lấy mã nguồn và khai báo cấu hình

Cell có thể chạy lại: nếu repo đã tồn tại thì không clone lần nữa.

In [ ]:
WORK = Path('/kaggle/working')
REPO = WORK / 'Low-Light-Image-Enhancement-and-Downstream-Recognition'
REPO_URL = 'https://github.com/NMH1203/Low-Light-Image-Enhancement-and-Downstream-Recognition.git'

if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)

RAW = REPO / 'Dataset' / 'exdark_yolo_dark'
CLAHE_ROOT = WORK / 'exdark_yolo_clahe_only'
FINAL_ROOT = WORK / 'exdark_yolo_clahe_denoised'
DENOISE_SCRIPT = REPO / 'src' / 'create_denoised_dataset.py'
RUNS = WORK / 'runs_nmh'

MODEL_NAME = 'yolov8n.pt'
EPOCHS = 40
IMAGE_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 15
SEED = 42

CFG = {
    'clahe_clip_limit': 2.0,
    'clahe_tile_grid': (8, 8),
    'denoise_diameter': 5,
    'denoise_sigma_color': 25.0,
    'denoise_sigma_space': 25.0,
    'model_name': MODEL_NAME,
    'epochs': EPOCHS,
    'batch': BATCH_SIZE,
    'imgsz': IMAGE_SIZE,
    'patience': PATIENCE,
    'seed': SEED,
}

assert RAW.is_dir(), f'Không tìm thấy dataset: {RAW}'
assert DENOISE_SCRIPT.is_file(), f'Không tìm thấy script: {DENOISE_SCRIPT}'
print(json.dumps(CFG, indent=2))

## 2. Tạo dataset CLAHE-only

CLAHE chỉ tác động lên kênh sáng `L` trong CIE LAB để hạn chế sai màu. Cell này giữ nguyên split và sao chép nhãn YOLO. Bilateral chưa được áp dụng ở đây để tránh khử nhiễu hai lần.

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SPLITS = {'train': 'train', 'val': 'valid', 'test': 'test'}

def load_data_yaml(root):
    with open(root / 'data.yaml', encoding='utf-8-sig') as f:
        return yaml.safe_load(f)

def locate_images(root, config, split):
    aliases = ('val', 'valid') if split == 'val' else (split,)
    candidates = []
    configured = config.get(split)
    if isinstance(configured, str):
        p = Path(configured)
        candidates.append(p if p.is_absolute() else root / p)
    for alias in aliases:
        candidates.extend([root / alias / 'images', root / 'images' / alias])
    return next((p for p in candidates if p.is_dir()), None)

def labels_for(root, image_dir):
    parts = list(image_dir.relative_to(root).parts)
    parts[parts.index('images')] = 'labels'
    return root.joinpath(*parts)

def clahe_lab(image_bgr, clip_limit=2.0, tile_grid=(8, 8)):
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    l_chan, a_chan, b_chan = cv2.split(lab)
    l_out = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid).apply(l_chan)
    return cv2.cvtColor(cv2.merge((l_out, a_chan, b_chan)), cv2.COLOR_LAB2BGR)

def build_clahe_dataset(source, output):
    if output.exists():
        shutil.rmtree(output)
    config = load_data_yaml(source)
    counts = {}
    for split, out_split in SPLITS.items():
        in_images = locate_images(source, config, split)
        if in_images is None:
            raise FileNotFoundError(f'Không tìm thấy ảnh split {split}')
        in_labels = labels_for(source, in_images)
        out_images = output / out_split / 'images'
        out_labels = output / out_split / 'labels'
        out_images.mkdir(parents=True)
        out_labels.mkdir(parents=True)
        files = sorted(p for p in in_images.iterdir() if p.suffix.lower() in IMAGE_EXTS)
        for image_path in tqdm(files, desc=f'CLAHE {split}'):
            image = cv2.imread(str(image_path))
            if image is None:
                raise ValueError(f'Không đọc được {image_path}')
            label_path = in_labels / f'{image_path.stem}.txt'
            if not label_path.is_file():
                raise FileNotFoundError(label_path)
            enhanced = clahe_lab(image, CFG['clahe_clip_limit'], CFG['clahe_tile_grid'])
            params = [cv2.IMWRITE_JPEG_QUALITY, 95] if image_path.suffix.lower() in {'.jpg', '.jpeg'} else []
            if not cv2.imwrite(str(out_images / image_path.name), enhanced, params):
                raise RuntimeError(f'Không ghi được {image_path.name}')
            shutil.copy2(label_path, out_labels / label_path.name)
        counts[split] = len(files)

    out_yaml = {
        'path': str(output), 'train': 'train/images', 'val': 'valid/images',
        'test': 'test/images', 'names': config['names']
    }
    with open(output / 'data.yaml', 'w', encoding='utf-8') as f:
        yaml.safe_dump(out_yaml, f, sort_keys=False, allow_unicode=True)
    return counts

counts = build_clahe_dataset(RAW, CLAHE_ROOT)
print('Đã tạo CLAHE:', counts)

## 3. Khử nhiễu bằng file có sẵn trên GitHub

Không chép lại thuật toán: notebook gọi trực tiếp `src/create_denoised_dataset.py`. Script dùng bilateral filter, giữ nguyên nhãn và sinh metadata để truy vết tham số.

In [ ]:
if FINAL_ROOT.exists():
    shutil.rmtree(FINAL_ROOT)

cmd = [
    sys.executable, str(DENOISE_SCRIPT),
    '--source', str(CLAHE_ROOT),
    '--output', str(FINAL_ROOT),
    '--diameter', str(CFG['denoise_diameter']),
    '--sigma-color', str(CFG['denoise_sigma_color']),
    '--sigma-space', str(CFG['denoise_sigma_space']),
    '--workers', str(min(8, os.cpu_count() or 2)),
]
subprocess.run(cmd, check=True)
print((FINAL_ROOT / 'denoising_metadata.json').read_text())

## 4. Kiểm tra định lượng và trực quan

Variance of Laplacian là chỉ báo độ sắc nét tương đối; độ lệch chuẩn kênh L mô tả tương phản. Hai chỉ số này chỉ dùng kiểm tra pipeline, không thay thế mAP.

In [ ]:
def image_stats(bgr):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    l_chan = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)[:, :, 0]
    return {'brightness_L': float(l_chan.mean()), 'contrast_L': float(l_chan.std()), 'lap_var': float(cv2.Laplacian(gray, cv2.CV_64F).var())}

raw_cfg = load_data_yaml(RAW)
raw_val = locate_images(RAW, raw_cfg, 'val')
sample = sorted(p for p in raw_val.iterdir() if p.suffix.lower() in IMAGE_EXTS)[0]
raw_img = cv2.imread(str(sample))
clahe_img = cv2.imread(str(CLAHE_ROOT / 'valid' / 'images' / sample.name))
final_img = cv2.imread(str(FINAL_ROOT / 'valid' / 'images' / sample.name))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, title, image in zip(axes, ['Ảnh tối gốc', 'CLAHE', 'CLAHE + denoise'], [raw_img, clahe_img, final_img]):
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{title}\n{image_stats(image)}', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Train YOLOv8n

Chỉ cấu hình CLAHE lấy theo NMH; thông số YOLO dùng bộ cấu hình được chỉ định ở đầu notebook.

In [ ]:
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else 'cpu'
model = YOLO(MODEL_NAME)
train_results = model.train(
    data=str(FINAL_ROOT / 'data.yaml'),
    epochs=CFG['epochs'],
    batch=CFG['batch'],
    imgsz=CFG['imgsz'],
    patience=CFG['patience'],
    device=device,
    seed=CFG['seed'],
    deterministic=True,
    project=str(RUNS),
    name='clahe_denoise_yolov8n_e40',
    exist_ok=True,
    plots=True,
)

## 6. Đánh giá test và lưu kết quả

Các số chính cần báo cáo: Precision, Recall, mAP@0.5 và mAP@0.5:0.95.

In [ ]:
BEST = RUNS / 'clahe_denoise_yolov8n_e40' / 'weights' / 'best.pt'
assert BEST.is_file(), f'Không thấy best.pt: {BEST}'
best_model = YOLO(str(BEST))
metrics = best_model.val(
    data=str(FINAL_ROOT / 'data.yaml'),
    split='test',
    imgsz=CFG['imgsz'],
    batch=CFG['batch'],
    device=device,
    project=str(RUNS),
    name='clahe_denoise_test',
    plots=True,
)

summary = {
    'pipeline': 'CLAHE(LAB-L) -> bilateral denoise -> YOLOv8n',
    'config': CFG,
    'precision': float(metrics.box.mp),
    'recall': float(metrics.box.mr),
    'mAP50': float(metrics.box.map50),
    'mAP50_95': float(metrics.box.map),
    'best_weights': str(BEST),
}
RESULT_JSON = WORK / 'clahe_denoise_nmh_results.json'
RESULT_JSON.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))

## 7. Xem dự đoán mẫu

In [ ]:
test_images = sorted((FINAL_ROOT / 'test' / 'images').glob('*'))[:6]
predictions = best_model.predict(test_images, imgsz=CFG['imgsz'], conf=0.25, device=device, verbose=False)
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, pred in zip(axes.ravel(), predictions):
    plotted_bgr = pred.plot()
    ax.imshow(cv2.cvtColor(plotted_bgr, cv2.COLOR_BGR2RGB))
    ax.set_title(Path(pred.path).name)
    ax.axis('off')
for ax in axes.ravel()[len(predictions):]:
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Weights:', BEST)
print('Metrics:', RESULT_JSON)